### In Class Naive Bayes — Irrigation Need 

In [16]:
# importing libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, classification_report
)

### Importing, inspecting, target encoding & splitting data

In [2]:
# load irrigation data, inspect it, encode target, and split it
irr_data=pd.read_csv(
    r"C:\Users\ldcal\OneDrive\Desktop\Cal Poly Courses\Spring\GSB-S545\Advanced_ML_Work\Git_Code\Kaggle Homework Lucas lucascalaff206\train.csv"
)

irr_data["Irrigation_Need"]=irr_data["Irrigation_Need"].map({
    "Low":0,
    "Medium":1,
    "High":2
})

print(irr_data.head())
print()
print(irr_data["Irrigation_Need"].value_counts().rename({0:"Low",1:"Medium",2:"High"}))
print()
print("missing values:",irr_data.isna().sum().sum())

X_irr=irr_data.drop(["id","Irrigation_Need"],axis=1)
y_irr=irr_data["Irrigation_Need"]

X_train_irr,X_test_irr,y_train_irr,y_test_irr=train_test_split(
    X_irr,y_irr,test_size=0.20,stratify=y_irr,random_state=42
)

   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy     4.92          32.58            1.01   
1   1      Clay     7.08          56.61            0.44   
2   2      Clay     5.69          27.71            0.81   
3   3     Sandy     5.65          13.32            1.33   
4   4      Clay     7.96          59.14            0.38   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     3.05          15.01     50.61       725.99   
1                     2.00          22.92     67.86       985.66   
2                     2.83          26.97     92.22      2201.70   
3                     0.87          13.32     61.57      1357.33   
4                     0.96          20.22     91.11      1538.20   

   Sunlight_Hours  ...  Crop_Type Crop_Growth_Stage  Season Irrigation_Type  \
0            5.90  ...  Sugarcane            Sowing    Zaid            Drip   
1            6.98  ...      Wheat        Vegetative  Kharif         Rainfed   

### Defining functions

In [3]:
# helper function for cross validation comparison (multiclass: macro averages)
def compare_models_cv(models,X_train,y_train,cv):
    cv_results=[]
    for name,model in models.items():
        scores=cross_validate(
            model,X_train,y_train,cv=cv,
            scoring=["accuracy","precision_macro","recall_macro","f1_macro","balanced_accuracy"]
        )
        cv_results.append({
            "model":name,
            "cv_accuracy":scores["test_accuracy"].mean(),
            "cv_precision_macro":scores["test_precision_macro"].mean(),
            "cv_recall_macro":scores["test_recall_macro"].mean(),
            "cv_f1_macro":scores["test_f1_macro"].mean(),
            "cv_balanced_accuracy":scores["test_balanced_accuracy"].mean()
        })
    return pd.DataFrame(cv_results).sort_values("cv_balanced_accuracy",ascending=False)

In [4]:
# helper function for threshold tuning on a chosen class in a multiclass setting
def evaluate_thresholds_multiclass(y_true,probs,target_class,thresholds=None):
    if thresholds is None:
        thresholds=np.linspace(0.05,0.60,23)
    rows=[]
    for t in thresholds:
        preds_t=probs.argmax(axis=1).copy()
        preds_t[probs[:,target_class]>=t]=target_class
        per_class_f1=f1_score(y_true,preds_t,average=None,zero_division=0)
        per_class_recall=recall_score(y_true,preds_t,average=None,zero_division=0)
        per_class_precision=precision_score(y_true,preds_t,average=None,zero_division=0)
        rows.append({
            "threshold":t,
            "f1_macro":f1_score(y_true,preds_t,average="macro",zero_division=0),
            "balanced_accuracy":balanced_accuracy_score(y_true,preds_t),
            "f1_high":per_class_f1[target_class],
            "recall_high":per_class_recall[target_class],
            "precision_high":per_class_precision[target_class]
        })
    return pd.DataFrame(rows)

In [5]:
# helper function for held out test evaluation (multiclass)
def evaluate_models_test(models,X_train,X_test,y_train,y_test):
    trained_models={}
    test_results=[]
    for name,model in models.items():
        model.fit(X_train,y_train)
        trained_models[name]=model
        preds=model.predict(X_test)
        test_results.append({
            "model":name,
            "test_accuracy":accuracy_score(y_test,preds),
            "test_precision_macro":precision_score(y_test,preds,average="macro",zero_division=0),
            "test_recall_macro":recall_score(y_test,preds,average="macro",zero_division=0),
            "test_f1_macro":f1_score(y_test,preds,average="macro",zero_division=0),
            "test_balanced_accuracy":balanced_accuracy_score(y_test,preds)
        })
    return trained_models,pd.DataFrame(test_results).sort_values("test_balanced_accuracy",ascending=False)

In [6]:
# helper for converting sparse matrices to dense arrays
def to_dense(x):
    return x.toarray() if hasattr(x,"toarray") else x

### Identifying categorical/numerical & defining CV

In [7]:
# identify numeric and categorical columns and define cross validation
numeric_features_irr=X_train_irr.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features_irr=X_train_irr.select_dtypes(include=["object","str"]).columns.tolist()

print("numeric columns:",numeric_features_irr)
print("categorical columns:",categorical_features_irr)

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

numeric columns: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


### Defining preprocessors & model

In [21]:
# define preprocessor and Naive Bayes model pipeline
numeric_transformer=Pipeline([
    ("imputer",SimpleImputer(strategy="median"))
])

categorical_transformer=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_irr=ColumnTransformer([
    ("num",numeric_transformer,numeric_features_irr),
    ("cat",categorical_transformer,categorical_features_irr)
])

models_irr={
    "Naive Bayes":Pipeline([
        ("preprocessor",preprocessor_irr),
        ("to_dense",FunctionTransformer(to_dense)),
        ("model",GaussianNB())
    ])
}

### Cross validation

In [22]:
# compare irrigation models with cross validation
compare_models_cv(models_irr,X_train_irr,y_train_irr,cv)

,model,cv_accuracy,cv_precision_macro,cv_recall_macro,cv_f1_macro,cv_balanced_accuracy
0,Naive Bayes,0.784417,0.702199,0.778128,0.731876,0.778128


### Baseline model evaluation on test data

In [10]:
# fit model on full training data and evaluate on held-out test set
trained_models_irr,test_results_irr=evaluate_models_test(
    models_irr,X_train_irr,X_test_irr,y_train_irr,y_test_irr
)
test_results_irr

,model,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_balanced_accuracy
0,Naive Bayes,0.784127,0.703626,0.779672,0.733473,0.779672


### Generating predicted probabilities

In [11]:
# generate predicted probabilities for all three classes
model_nb=trained_models_irr["Naive Bayes"]
probs_irr_nb=model_nb.predict_proba(X_test_irr)

# class order: 0=Low, 1=Medium, 2=High
print("class order (model.classes_):",model_nb.named_steps["model"].classes_)
print("proba sample (first 5 rows):")
print(pd.DataFrame(probs_irr_nb[:5],columns=["P(Low)","P(Medium)","P(High)"]))

class order (model.classes_): [0 1 2]
proba sample (first 5 rows):
     P(Low)  P(Medium)       P(High)
0  0.000270   0.079679  9.200513e-01
1  0.994810   0.005190  2.111891e-23
2  0.996486   0.003514  9.180874e-23
3  0.001509   0.226497  7.719939e-01
4  0.995892   0.004108  1.833121e-12


### Baseline predictions — default rule (highest probability class)

In [12]:
# baseline predictions: assign whichever class has the highest predicted probability
preds_baseline=probs_irr_nb.argmax(axis=1)

print("Baseline predictions — default rule (threshold=0.5 equivalent: argmax)")
print()
print("balanced accuracy:",round(balanced_accuracy_score(y_test_irr,preds_baseline),4))
print("f1 macro:",round(f1_score(y_test_irr,preds_baseline,average="macro",zero_division=0),4))
print()
print(classification_report(y_test_irr,preds_baseline,target_names=["Low","Medium","High"]))

Baseline predictions — default rule (threshold=0.5 equivalent: argmax)

balanced accuracy: 0.7797
f1 macro: 0.7335

              precision    recall  f1-score   support

         Low       0.87      0.80      0.83     73983
      Medium       0.70      0.76      0.73     47815
        High       0.54      0.78      0.64      4202

    accuracy                           0.78    126000
   macro avg       0.70      0.78      0.73    126000
weighted avg       0.80      0.78      0.79    126000



### Threshold tuning — focusing on the "High" irrigation class

High is the rarest class (~3.3% of data). The model naturally underpredicts it.
We lower the threshold so that any sample with P(High) >= threshold is assigned High,
overriding the argmax decision. We track F1 for the High class specifically.

In [13]:
# evaluate thresholds for the High class (class index 2)
high_class=2
threshold_results_irr=evaluate_thresholds_multiclass(y_test_irr,probs_irr_nb,target_class=high_class)

best=threshold_results_irr.sort_values("f1_high",ascending=False).iloc[0]

print("Top thresholds by High-class F1:")
print(threshold_results_irr.sort_values("f1_high",ascending=False).head(8))
print()
print("baseline (argmax):")
print("  f1_high:",round(f1_score(y_test_irr,preds_baseline,average=None,zero_division=0)[high_class],4))
print("  recall_high:",round(recall_score(y_test_irr,preds_baseline,average=None,zero_division=0)[high_class],4))
print("  balanced_accuracy:",round(balanced_accuracy_score(y_test_irr,preds_baseline),4))
print()
print("best threshold:",round(best["threshold"],4))
print("best f1_high:",round(best["f1_high"],4))
print("recall_high at best threshold:",round(best["recall_high"],4))
print("balanced_accuracy at best threshold:",round(best["balanced_accuracy"],4))

Top thresholds by High-class F1:
    threshold  f1_macro  balanced_accuracy   f1_high  recall_high  \
21      0.575  0.733473           0.779672  0.637526     0.777487   
20      0.550  0.733473           0.779672  0.637526     0.777487   
18      0.500  0.733473           0.779672  0.637526     0.777487   
22      0.600  0.733473           0.779672  0.637526     0.777487   
19      0.525  0.733473           0.779672  0.637526     0.777487   
17      0.475  0.729260           0.779409  0.627282     0.780819   
16      0.450  0.725467           0.780022  0.618362     0.787006   
15      0.425  0.721399           0.780123  0.608727     0.791766   

    precision_high  
21        0.540268  
20        0.540268  
18        0.540268  
22        0.540268  
19        0.540268  
17        0.524205  
16        0.509239  
15        0.494427  

baseline (argmax):
  f1_high: 0.6375
  recall_high: 0.7775
  balanced_accuracy: 0.7797

best threshold: 0.575
best f1_high: 0.6375
recall_high at best thre

In [14]:
# apply chosen threshold to generate second set of predictions
chosen_threshold=round(float(best["threshold"]),2)

preds_threshold=preds_baseline.copy()
preds_threshold[probs_irr_nb[:,high_class]>=chosen_threshold]=high_class

print(f"Threshold predictions — P(High) >= {chosen_threshold}")
print()
print("balanced accuracy:",round(balanced_accuracy_score(y_test_irr,preds_threshold),4))
print("f1 macro:",round(f1_score(y_test_irr,preds_threshold,average="macro",zero_division=0),4))
print()
print(classification_report(y_test_irr,preds_threshold,target_names=["Low","Medium","High"]))

Threshold predictions — P(High) >= 0.57

balanced accuracy: 0.7797
f1 macro: 0.7335

              precision    recall  f1-score   support

         Low       0.87      0.80      0.83     73983
      Medium       0.70      0.76      0.73     47815
        High       0.54      0.78      0.64      4202

    accuracy                           0.78    126000
   macro avg       0.70      0.78      0.73    126000
weighted avg       0.80      0.78      0.79    126000



### Comparing baseline vs threshold predictions

In [15]:
# side-by-side summary: baseline (argmax) vs threshold-tuned predictions
baseline_per_class_f1=f1_score(y_test_irr,preds_baseline,average=None,zero_division=0)
threshold_per_class_f1=f1_score(y_test_irr,preds_threshold,average=None,zero_division=0)

pd.DataFrame([
    {
        "method":"baseline (argmax)",
        "threshold":"none",
        "f1_Low":round(baseline_per_class_f1[0],4),
        "f1_Medium":round(baseline_per_class_f1[1],4),
        "f1_High":round(baseline_per_class_f1[2],4),
        "f1_macro":round(f1_score(y_test_irr,preds_baseline,average="macro",zero_division=0),4),
        "balanced_accuracy":round(balanced_accuracy_score(y_test_irr,preds_baseline),4)
    },
    {
        "method":f"threshold (P(High)>={chosen_threshold})",
        "threshold":chosen_threshold,
        "f1_Low":round(threshold_per_class_f1[0],4),
        "f1_Medium":round(threshold_per_class_f1[1],4),
        "f1_High":round(threshold_per_class_f1[2],4),
        "f1_macro":round(f1_score(y_test_irr,preds_threshold,average="macro",zero_division=0),4),
        "balanced_accuracy":round(balanced_accuracy_score(y_test_irr,preds_threshold),4)
    }
])

,method,threshold,f1_Low,f1_Medium,f1_High,f1_macro,balanced_accuracy
0,baseline (argmax),none,0.8333,0.7296,0.6375,0.7335,0.7797
1,threshold (P(High)>=0.57),0.57,0.8333,0.7296,0.6375,0.7335,0.7797


## Interpretation 

The class that we selected to use was the high class as this was the rarest class and wanted to observe if by manipualting our thershold we could gain better predictability. The threshold that ended up being choosen was 0.57 which isnt very far off from the base models .5, which goes to show why our metrics dont change from the base model to the tuned one. naive does not perform as well as the other models but for a baseline model does not do horribly in predicting these classes, its likely that having such a skewed dataset did not help when trying to average out features to gain insight on how to classify. This model is likely better used on data that isn't as lopsided as the irrigation dataset we have been working for, which likely needs more advanced models to gain proper insights. 